# LSTM Implementation

First, let's import some libraries and make sure the runtime has access to a GPU.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Select the Runtime > "Change runtime type" menu to enable a GPU accelerator, ')
    print('and then re-execute this cell.')
else:
    print(gpu_info)

print(f'GPU available: {torch.cuda.is_available()}')

Thu Mar 27 08:48:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Download the Data

Run the following code to download the English part of the CoNLL 2003 dataset, the evaluation script and pre-filtered GloVe embeddings we are providing for this data.

In [3]:
#CoNLL 2003 data
!wget https://raw.githubusercontent.com/patverga/torch-ner-nlp-from-scratch/master/data/conll2003/eng.train
!wget https://raw.githubusercontent.com/patverga/torch-ner-nlp-from-scratch/master/data/conll2003/eng.testa
!wget https://raw.githubusercontent.com/patverga/torch-ner-nlp-from-scratch/master/data/conll2003/eng.testb
!cat eng.train | awk '{print $1 "\t" $4}' > train
!cat eng.testa | awk '{print $1 "\t" $4}' > dev
!cat eng.testb | awk '{print $1 "\t" $4}' > test

#Evaluation Script
!wget https://raw.githubusercontent.com/aritter/twitter_nlp/master/data/annotated/wnut16/conlleval.pl

#Pre-filtered GloVe embeddings
!wget https://raw.githubusercontent.com/aritter/aritter.github.io/master/files/glove.840B.300d.conll_filtered.txt

--2025-03-27 08:48:37--  https://raw.githubusercontent.com/patverga/torch-ner-nlp-from-scratch/master/data/conll2003/eng.train
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3283420 (3.1M) [text/plain]
Saving to: ‘eng.train’

eng.train           100%[===================>]   3.13M  --.-KB/s    in 0.03s   

2025-03-27 08:48:38 (102 MB/s) - ‘eng.train’ saved [3283420/3283420]

--2025-03-27 08:48:38--  https://raw.githubusercontent.com/patverga/torch-ner-nlp-from-scratch/master/data/conll2003/eng.testa
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response..

## CoNLL Data Format

Run the following cell to see a sample of the data in CoNLL format.  As you can see, each line in the file represents a word and its labeled named entity tag in BIO format.  A blank line is used to seperate sentences.

In [4]:
!head -n 20 train

-DOCSTART-	O
	
EU	I-ORG
rejects	O
German	I-MISC
call	O
to	O
boycott	O
British	I-MISC
lamb	O
.	O
	
Peter	I-PER
Blackburn	I-PER
	
BRUSSELS	I-LOC
1996-08-22	O
	
The	O
European	I-ORG


## Reading in the Data

Below we proivide a bit of code to read in data in the CoNLL format.  This also reads in the filtered GloVe embeddings, to save you some effort - we will discuss this more later.

In [5]:
#Read in the training data
def read_conll_format(filename):
    (words, tags, currentSent, currentTags) = ([],[],['-START-'],['START'])
    for line in open(filename).readlines():
        line = line.strip()
        #print(line)
        if line == "":
            currentSent.append('-END-')
            currentTags.append('END')
            words.append(currentSent)
            tags.append(currentTags)
            (currentSent, currentTags) = (['-START-'], ['START'])
        else:
            (word, tag) = line.split()
            currentSent.append(word)
            currentTags.append(tag)
    return (words, tags)

def sentences2char(sentences):
    return [[['start'] + [c for c in w] + ['end'] for w in l] for l in sentences]


(sentences_train, tags_train) = read_conll_format("train")
(sentences_dev, tags_dev)     = read_conll_format("dev")

print(sentences_train[2])
print(tags_train[2])

sentencesChar = sentences2char(sentences_train)

print(sentencesChar[2])

['-START-', 'Peter', 'Blackburn', '-END-']
['START', 'I-PER', 'I-PER', 'END']
[['start', '-', 'S', 'T', 'A', 'R', 'T', '-', 'end'], ['start', 'P', 'e', 't', 'e', 'r', 'end'], ['start', 'B', 'l', 'a', 'c', 'k', 'b', 'u', 'r', 'n', 'end'], ['start', '-', 'E', 'N', 'D', '-', 'end']]


In [6]:
#Read GloVe embeddings.
def read_GloVe(filename):
    embeddings = {}
    for line in open(filename).readlines():
        #print(line)
        fields = line.strip().split(" ")
        word = fields[0]
        embeddings[word] = [float(x) for x in fields[1:]]
    return embeddings

GloVe = read_GloVe("glove.840B.300d.conll_filtered.txt")

print(GloVe["the"])
print("dimension of glove embedding:", len(GloVe["the"]))

[0.27204, -0.06203, -0.1884, 0.023225, -0.018158, 0.0067192, -0.13877, 0.17708, 0.17709, 2.5882, -0.35179, -0.17312, 0.43285, -0.10708, 0.15006, -0.19982, -0.19093, 1.1871, -0.16207, -0.23538, 0.003664, -0.19156, -0.085662, 0.039199, -0.066449, -0.04209, -0.19122, 0.011679, -0.37138, 0.21886, 0.0011423, 0.4319, -0.14205, 0.38059, 0.30654, 0.020167, -0.18316, -0.0065186, -0.0080549, -0.12063, 0.027507, 0.29839, -0.22896, -0.22882, 0.14671, -0.076301, -0.1268, -0.0066651, -0.052795, 0.14258, 0.1561, 0.05551, -0.16149, 0.09629, -0.076533, -0.049971, -0.010195, -0.047641, -0.16679, -0.2394, 0.0050141, -0.049175, 0.013338, 0.41923, -0.10104, 0.015111, -0.077706, -0.13471, 0.119, 0.10802, 0.21061, -0.051904, 0.18527, 0.17856, 0.041293, -0.014385, -0.082567, -0.035483, -0.076173, -0.045367, 0.089281, 0.33672, -0.22099, -0.0067275, 0.23983, -0.23147, -0.88592, 0.091297, -0.012123, 0.013233, -0.25799, -0.02972, 0.016754, 0.01369, 0.32377, 0.039546, 0.042114, -0.088243, 0.30318, 0.087747, 0.1634

## Mapping Tokens to Indices

As in the last project, we will need to convert words in the dataset to numeric indices, so they can be presented as input to a neural network.  Code to handle this for you with sample usage is provided below.

In [7]:
#Create mappings between tokens and indices.

from collections import Counter
import random

#Will need this later to remove 50% of words that only appear once in the training data from the vocabulary (and don't have GloVe embeddings).
wordCounts = Counter([w for l in sentences_train for w in l])
charCounts = Counter([c for l in sentences_train for w in l for c in w])
singletons = set([w for (w,c) in wordCounts.items() if c == 1 and not w in GloVe.keys()])
charSingletons = set([w for (w,c) in charCounts.items() if c == 1])

#Build dictionaries to map from words, characters to indices and vice versa.
#Save first two words in the vocabulary for padding and "UNK" token.
word2i = {w:i+2 for i,w in enumerate(set([w for l in sentences_train for w in l] + list(GloVe.keys())))}
char2i = {w:i+2 for i,w in enumerate(set([c for l in sentencesChar for w in l for c in w]))}
i2word = {i:w for w,i in word2i.items()}
i2char = {i:w for w,i in char2i.items()}

vocab_size = max(word2i.values()) + 1
char_vocab_size = max(char2i.values()) + 1

#Tag dictionaries.
tag2i = {w:i for i,w in enumerate(set([t for l in tags_train for t in l]))}
i2tag = {i:t for t,i in tag2i.items()}

#When training, randomly replace singletons with UNK tokens sometimes to simulate situation at test time.
def getDictionaryRandomUnk(w, dictionary, train=False):
    if train and (w in singletons and random.random() > 0.5):
        return 1
    else:
        return dictionary.get(w, 1)

#Map a list of sentences from words to indices.
def sentences2indices(words, dictionary, train=False):
    #1.0 => UNK
    return [[getDictionaryRandomUnk(w,dictionary, train=train) for w in l] for l in words]

#Map a list of sentences containing to indices (character indices)
def sentences2indicesChar(chars, dictionary):
    #1.0 => UNK
    return [[[dictionary.get(c,1) for c in w] for w in l] for l in chars]

#Indices
X       = sentences2indices(sentences_train, word2i, train=True)
X_char  = sentences2indicesChar(sentencesChar, char2i)
Y       = sentences2indices(tags_train, tag2i)

print("vocab size:", vocab_size)
print("char vocab size:", char_vocab_size)
print()

print("index of word 'the':", word2i["the"])
print("word of index 253:", i2word[253])
print()

#Print out some examples of what the dev inputs will look like
for i in range(10):
    print(" ".join([i2word.get(w,'UNK') for w in X[i]]))

vocab size: 29148
char vocab size: 88

index of word 'the': 17671
word of index 253: 22.68

-START- -DOCSTART- -END-
-START- EU rejects German call to boycott British lamb . -END-
-START- Peter Blackburn -END-
-START- BRUSSELS 1996-08-22 -END-
-START- The European Commission said on Thursday it disagreed with German advice to consumers to shun British lamb until scientists determine whether mad cow disease can be transmitted to sheep . -END-
-START- Germany 's representative to the European Union 's veterinary committee Werner Zwingmann said on Wednesday consumers should buy sheepmeat from countries other than Britain until the scientific advice was clearer . -END-
-START- " We do n't support any such recommendation because we do n't see any grounds for it , " the Commission 's chief spokesman Nikolaus van der Pas told a news briefing . -END-
-START- He said further scientific study was required and if it was found that action was needed it should be taken by the European Union . -END-

## Padding and Batching

In this assignment, you should train your models using minibatched SGD, rather than using a batch size of 1 as we did in the previous project.  When presenting multiple sentences to the network at the same time, we will need to pad them to be of the same length. We use [torch.nn.utils.rnn.pad_sequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html) to do so.

Below we provide some code to prepare batches of data to present to the network.  pad the sequence so that all the sequences have the same length.

**Side Note:** PyTorch includes utilities in [`torch.utils.data`](https://pytorch.org/docs/stable/data.html) to help with padding, batching, shuffling and some other things, but for this assignment we will do everything from scratch to help you see exactly how this works.

In [8]:
#Pad inputs to max sequence length (for batching)
def prepare_input(X_list):
    X_padded = torch.nn.utils.rnn.pad_sequence([torch.as_tensor(l) for l in X_list], batch_first=True).type(torch.LongTensor) # padding the sequences with 0
    X_mask   = torch.nn.utils.rnn.pad_sequence([torch.as_tensor([1.0] * len(l)) for l in X_list], batch_first=True).type(torch.FloatTensor) # consisting of 0 and 1, 0 for padded positions, 1 for non-padded positions
    return (X_padded, X_mask)

#Maximum word length (for character representations)
MAX_CLEN=32

def prepare_input_char(X_list):
    MAX_SLEN = max([len(l) for l in X_list])
    X_padded  = [l + [[]]*(MAX_SLEN-len(l))  for l in X_list]
    X_padded  = [[w[0:MAX_CLEN] for w in l] for l in X_padded]
    X_padded  = [[w + [1]*(MAX_CLEN-len(w)) for w in l] for l in X_padded]
    return torch.as_tensor(X_padded).type(torch.LongTensor)

#Pad outputs using one-hot encoding
def prepare_output_onehot(Y_list, NUM_TAGS=max(tag2i.values())+1):
    Y_onehot = [torch.zeros(len(l), NUM_TAGS) for l in Y_list]
    for i in range(len(Y_list)):
        for j in range(len(Y_list[i])):
            Y_onehot[i][j,Y_list[i][j]] = 1.0
    Y_padded = torch.nn.utils.rnn.pad_sequence(Y_onehot, batch_first=True).type(torch.FloatTensor)
    return Y_padded

print("max slen:", max([len(x) for x in X_char]))  #Max sequence length in the training data is 39.

(X_padded, X_mask) = prepare_input(X)
X_padded_char      = prepare_input_char(X_char)
Y_onehot           = prepare_output_onehot(Y)

print("X_padded:", X_padded.shape)
print("X_mask:", X_mask.shape)
print("X_padded_char:", X_padded_char.shape)
print("Y_onehot:", Y_onehot.shape)

max slen: 115
X_padded: torch.Size([14987, 115])
X_mask: torch.Size([14987, 115])
X_padded_char: torch.Size([14987, 115, 32])
Y_onehot: torch.Size([14987, 115, 10])


## **Your code starts here:** Basic LSTM Tagger (10 points)

OK, now you should have everything you need to get started.

Recall that your goal is to to implement the BiLSTM-CNN-CRF, as described in [(Ma and Hovy, 2016)](https://www.aclweb.org/anthology/P16-1101.pdf).  This is a relatively complex network with various components.  Below we provide starter code to break down your implementation into increasingly complex versions of the final model, starting with a Basic LSTM tagger.  This way you can be confident that each part is working correctly before incrementally increasing the complexity of your implementation.  This is generally a good approach to take when implementing complex models, since buggy PyTorch code is often partially working, but produces worse results than a correct implementation, so it's hard to know whether added complexities are helping or hurting.  Also, if you aren't able to match published results it's hard to know which component of your model has the problem (or even whether or not it is a problem in the published result!)

Fill in the functions marked as `TODO` in the code block below.  If everything is working correctly, you should be able to achieve an **F1 score of 0.87 on the dev set and 0.83 on the test set (with GloVe embeddings)**. You are required to initialize word embeddings with GloVe later, but you can randomly initialize the word embeddings in the beginning.

In [9]:
class BasicLSTMtagger(nn.Module):
    def __init__(self, DIM_EMB=10, DIM_HID=10):
        super(BasicLSTMtagger, self).__init__()
        NUM_TAGS = max(tag2i.values())+1

        (self.DIM_EMB, self.NUM_TAGS) = (DIM_EMB, NUM_TAGS)
        
        self.embedding = nn.Embedding(vocab_size, DIM_EMB, padding_idx=0)
        self.lstm = nn.LSTM(input_size=DIM_EMB, hidden_size=DIM_HID, batch_first=True, bidirectional=True)
        self.linear = nn.Linear(DIM_HID * 2, NUM_TAGS)
        self.log_softmax = nn.LogSoftmax(dim=2)

    def forward(self, X, train=False):
    
        embedded = self.embedding(X)
        lstm_output, _ = self.lstm(embedded)
        linear_output = self.linear(lstm_output)
        log_softmax_output = self.log_softmax(linear_output)

        return log_softmax_output

    def init_glove(self, GloVe):

        embeddings = torch.randn(vocab_size, self.DIM_EMB)

        for word, index in word2i.items():
            if word in GloVe:
                embeddings[index] = torch.tensor(GloVe[word])

        self.embedding.weight.data.copy_(embeddings)

    def inference(self, sentences):
        X       = prepare_input(sentences2indices(sentences, word2i))[0].cuda()
        pred = self.forward(X).argmax(dim=2)
        return [[i2tag[pred[i,j].item()] for j in range(len(sentences[i]))] for i in range(len(sentences))]

    def print_predictions(self, words, tags):
        Y_pred = self.inference(words)
        for i in range(len(words)):
            print("----------------------------")
            print(" ".join([f"{words[i][j]}/{Y_pred[i][j]}/{tags[i][j]}" for j in range(len(words[i]))]))
            print("Predicted:\t", Y_pred[i])
            print("Gold:\t\t", tags[i])

    def write_predictions(self, sentences, outFile):
        fOut = open(outFile, 'w')
        for s in sentences:
            y = self.inference([s])[0]
            #print("\n".join(y[1:len(y)-1]))
            fOut.write("\n".join(y[1:len(y)-1]))  #Skip start and end tokens
            fOut.write("\n\n")

#The following code will initialize a model and test that your forward computation runs without errors.
lstm_test   = BasicLSTMtagger(DIM_HID=7, DIM_EMB=300)
lstm_output = lstm_test.forward(prepare_input(X[0:5])[0])
Y_onehot    = prepare_output_onehot(Y[0:5])

#Check the shape of the lstm_output and one-hot label tensors.
print("lstm output shape:", lstm_output.shape)
print("Y onehot shape:", Y_onehot.shape)

lstm output shape: torch.Size([5, 32, 10])
Y onehot shape: torch.Size([5, 32, 10])


In [10]:
#Read in the data

(sentences_dev, tags_dev)     = read_conll_format('dev')
(sentences_train, tags_train) = read_conll_format('train')
(sentences_test, tags_test)   = read_conll_format('test')

# Train your Model (10 points)

Next, implement the function below to train your basic BiLSTM tagger.  See [torch.nn.lstm](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html).  Make sure to save your predictions on the test set (`test_pred_lstm.txt`) for submission to GradeScope. Feel free to change number of epochs, optimizer, learning rate and batch size.

In [68]:
from math import log
#Training

from random import sample
from tqdm.notebook import tqdm
import os
import subprocess
import random

def shuffle_sentences(sentences, tags):
    shuffled_sentences = []
    shuffled_tags      = []
    indices = list(range(len(sentences)))
    random.shuffle(indices)
    for i in indices:
        shuffled_sentences.append(sentences[i])
        shuffled_tags.append(tags[i])
    return (shuffled_sentences, shuffled_tags)

nEpochs = 20

def train_basic_lstm(sentences, tags, lstm):
    optimizer = optim.Adam(lstm.parameters(), lr=0.01)

    batchSize = 50
    
    loss_fn = torch.nn.NLLLoss()

    for epoch in range(nEpochs):
        totalLoss = 0.0

        (sentences_shuffled, tags_shuffled) = shuffle_sentences(sentences, tags)
        for batch in tqdm(range(0, len(sentences), batchSize), leave=False):
            lstm.zero_grad()
        
            batch_sentences = sentences_shuffled[batch:batch+batchSize]
    
            batch_tags      = tags_shuffled[batch:batch+batchSize]

            x_batch = prepare_input(sentences2indices(batch_sentences, word2i, train=True))[0].cuda()
        
            Y_gold = prepare_input(sentences2indices(batch_tags, tag2i))[0].cuda()

            log_probabilities = lstm.forward(x_batch)
            
            log_probabilities = log_probabilities.view(-1, log_probabilities.shape[2])
            Y_gold = Y_gold.view(-1)
            
            loss = loss_fn(log_probabilities, Y_gold)

            loss.backward()
            optimizer.step()
            
            totalLoss += loss.item()

        print(f"loss on epoch {epoch} = {totalLoss}")
        lstm.write_predictions(sentences_dev, 'dev_pred')   #Performance on dev set
        print('conlleval:')
        print(subprocess.Popen('paste dev dev_pred | perl conlleval.pl -d "\t"', shell=True, stdout=subprocess.PIPE,stderr=subprocess.STDOUT).communicate()[0].decode('UTF-8'))

        if epoch % 10 == 0:
            s = sample(range(len(sentences_dev)), 5)
            lstm.print_predictions([sentences_dev[i] for i in s], [tags_dev[i] for i in s])

lstm = BasicLSTMtagger(DIM_HID=500, DIM_EMB=300).cuda()
lstm.init_glove(GloVe)
train_basic_lstm(sentences_train, tags_train, lstm)

  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 0 = 22.040442353114486
conlleval:
processed 51578 tokens with 5942 phrases; found: 5948 phrases; correct: 5053.
accuracy:  97.65%; precision:  84.95%; recall:  85.04%; FB1:  85.00
              LOC: precision:  90.99%; recall:  91.78%; FB1:  91.38  1853
             MISC: precision:  84.54%; recall:  80.04%; FB1:  82.23  873
              ORG: precision:  76.91%; recall:  77.26%; FB1:  77.08  1347
              PER: precision:  84.96%; recall:  86.48%; FB1:  85.71  1875

----------------------------
-START-/START/START Berlin/I-LOC/I-LOC ,/O/O Brussels/I-LOC/I-LOC and/O/O Zurich/I-LOC/I-LOC all/O/O belong/O/O to/O/O the/O/O most/O/O lucrative/O/O series/O/O in/O/O the/O/O sport/O/O ,/O/O the/O/O Golden/I-MISC/I-MISC Four/I-MISC/I-MISC ./O/O -END-/END/END
Predicted:	 ['START', 'I-LOC', 'O', 'I-LOC', 'O', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-MISC', 'I-MISC', 'O', 'END']
Gold:		 ['START', 'I-LOC', 'O', 'I-LOC', 'O', 'I-LOC', 'O', 'O', 'O', 

  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 1 = 3.6641035210341215
conlleval:
processed 51578 tokens with 5942 phrases; found: 5974 phrases; correct: 5095.
accuracy:  97.77%; precision:  85.29%; recall:  85.75%; FB1:  85.52
              LOC: precision:  91.03%; recall:  91.67%; FB1:  91.35  1850
             MISC: precision:  78.31%; recall:  83.41%; FB1:  80.78  982
              ORG: precision:  82.06%; recall:  77.11%; FB1:  79.51  1260
              PER: precision:  85.44%; recall:  87.30%; FB1:  86.36  1882



  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 2 = 2.0307455671718344
conlleval:
processed 51578 tokens with 5942 phrases; found: 5988 phrases; correct: 5092.
accuracy:  97.73%; precision:  85.04%; recall:  85.70%; FB1:  85.36
              LOC: precision:  92.73%; recall:  88.13%; FB1:  90.37  1746
             MISC: precision:  81.65%; recall:  83.95%; FB1:  82.78  948
              ORG: precision:  75.80%; recall:  83.15%; FB1:  79.30  1471
              PER: precision:  86.89%; recall:  85.99%; FB1:  86.44  1823



... (30 outputs left out)

  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 18 = 0.7569827051047469
conlleval:
processed 51578 tokens with 5942 phrases; found: 6434 phrases; correct: 5043.
accuracy:  96.82%; precision:  78.38%; recall:  84.87%; FB1:  81.50
              LOC: precision:  85.71%; recall:  89.11%; FB1:  87.38  1910
             MISC: precision:  68.09%; recall:  83.30%; FB1:  74.93  1128
              ORG: precision:  73.10%; recall:  77.40%; FB1:  75.19  1420
              PER: precision:  80.97%; recall:  86.86%; FB1:  83.81  1976



  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 19 = 0.8922091168860788
conlleval:
processed 51578 tokens with 5942 phrases; found: 6174 phrases; correct: 4990.
accuracy:  97.08%; precision:  80.82%; recall:  83.98%; FB1:  82.37
              LOC: precision:  87.80%; recall:  89.71%; FB1:  88.75  1877
             MISC: precision:  75.77%; recall:  80.04%; FB1:  77.85  974
              ORG: precision:  75.47%; recall:  74.79%; FB1:  75.13  1329
              PER: precision:  80.29%; recall:  86.92%; FB1:  83.47  1994



In [69]:
#Evaluation on test data
lstm.init_glove(GloVe)
lstm.write_predictions(sentences_test, 'test_pred_lstm.txt')
!wget https://raw.githubusercontent.com/aritter/twitter_nlp/master/data/annotated/wnut16/conlleval.pl
!paste test test_pred_lstm.txt | perl conlleval.pl -d "\t"

--2025-03-27 10:47:34--  https://raw.githubusercontent.com/aritter/twitter_nlp/master/data/annotated/wnut16/conlleval.pl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12754 (12K) [text/plain]
Saving to: ‘conlleval.pl.15’

conlleval.pl.15     100%[===================>]  12.46K  --.-KB/s    in 0s      

2025-03-27 10:47:34 (122 MB/s) - ‘conlleval.pl.15’ saved [12754/12754]

processed 46666 tokens with 5648 phrases; found: 5086 phrases; correct: 3150.
accuracy:  92.06%; precision:  61.93%; recall:  55.77%; FB1:  58.69
              LOC: precision:  77.05%; recall:  60.97%; FB1:  68.07  1320
             MISC: precision:  32.81%; recall:  50.57%; FB1:  39.80  1082
              ORG: precision:  63.82%; recall:  46.84%; FB1:  54.03  1219
              PER: prec

## Initialization with GloVe Embeddings (5 points)

If you haven't already, implement the `init_glove()` method in `BasicLSTMtagger` above.

Rather than initializing word embeddings randomly, it is common to use learned word embeddings (GloVe or Word2Vec), as discussed in lecture.  To make this simpler, we have already pre-filtered [GloVe](https://nlp.stanford.edu/projects/glove/) embeddings to only contain words in the vocabulary of the CoNLL NER dataset, and loaded them into a dictionary (`GloVe`) at the beginning of this notebook.



## Character Embeddings (10 points)

Now that you have your basic LSTM tagger working, the next step is to add a convolutional network that computes word embeddings from character representations of words.  See Figure 2 and Figure 3 in the [Ma and Hovy](https://www.aclweb.org/anthology/P16-1101.pdf) paper.  We have provided code in `sentences2input_tensors` to convert sentences into lists of word and character indices.  See also [nn.Conv1d](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) and [MaxPool1d](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool1d.html).

Hint: The nn.Conv1d accepts input size $(N, C_{in}, L_{in})$, but we have input size $(N, \text{SLEN}, \text{CLEN}, \text{EMB_DIM})$. We can reshape and [permute](https://pytorch.org/docs/stable/generated/torch.permute.html) our input to satisfy the nn.Conv1d, and recover the dimensions later.

Make sure to save your predictions on the test set, for submission to GradeScope.  You should be able to achieve **90 F1 / 85 F1 on the dev/test sets**.

In [58]:
import torch.nn.functional as F


class CharLSTMtagger(BasicLSTMtagger):
    def __init__(self, DIM_EMB=10, DIM_CHAR_EMB=30, DIM_HID=10):
        super(CharLSTMtagger, self).__init__(DIM_EMB=DIM_EMB, DIM_HID=DIM_HID)
        NUM_TAGS = max(tag2i.values())+1

        (self.DIM_EMB, self.NUM_TAGS) = (DIM_EMB, NUM_TAGS)
        
        self.DIM_CHAR_EMB = DIM_CHAR_EMB
        self.DIM_HID = DIM_HID

        self.char_vocab_size = max(char2i.values()) + 1

        self.char_embedding = nn.Embedding(self.char_vocab_size, DIM_CHAR_EMB, padding_idx=0)

        self.char_conv = nn.Conv1d(in_channels=DIM_CHAR_EMB, out_channels=50, kernel_size=3, padding=1)

        self.lstm = nn.LSTM(input_size=DIM_EMB + 50, hidden_size=DIM_HID, batch_first=True, bidirectional=True)
        self.linear = nn.Linear(DIM_HID * 2, self.NUM_TAGS)
        self.log_softmax = nn.LogSoftmax(dim=2)

        char_lstm.embedding.weight.data.copy_(lstm.embedding.weight.data.clone())


    def forward(self, X, X_char, train=False):

        batch_size, sequence_len, char_len = X_char.shape

        char_embedded = self.char_embedding(X_char)

        char_embedded = char_embedded.view(-1, char_len, self.DIM_CHAR_EMB)
       
        char_embedded = char_embedded.permute(0, 2, 1)

        char_conv_output = self.char_conv(char_embedded)
        
        char_conv_output = F.relu(char_conv_output)
        
        char_pool_output = F.max_pool1d(char_conv_output, char_conv_output.shape[2]).squeeze(2)
        
        char_restore = char_pool_output.view(batch_size, sequence_len, -1)

        
        word_embedded = self.embedding(X)
        
        embedded_combined = torch.cat((word_embedded, char_restore), dim=2)

        lstm_output, _ = self.lstm(embedded_combined)
        linear_output = self.linear(lstm_output)
        log_softmax_output = self.log_softmax(linear_output)

        return log_softmax_output

    def sentences2input_tensors(self, sentences):
        (X, X_mask)   = prepare_input(sentences2indices(sentences, word2i))
        X_char        = prepare_input_char(sentences2indicesChar(sentences, char2i))
        return (X, X_mask, X_char)

    def inference(self, sentences):
        (X, X_mask, X_char) = self.sentences2input_tensors(sentences)
        pred = self.forward(X.cuda(), X_char.cuda()).argmax(dim=2)
        return [[i2tag[pred[i,j].item()] for j in range(len(sentences[i]))] for i in range(len(sentences))]

    def print_predictions(self, words, tags):
        Y_pred = self.inference(words)
        for i in range(len(words)):
            print("----------------------------")
            print(" ".join([f"{words[i][j]}/{Y_pred[i][j]}/{tags[i][j]}" for j in range(len(words[i]))]))
            print("Predicted:\t", Y_pred[i])
            print("Gold:\t\t", tags[i])

char_lstm_test = CharLSTMtagger(DIM_HID=7, DIM_EMB=300)
lstm_output    = char_lstm_test.forward(prepare_input(X[0:5])[0], prepare_input_char(X_char[0:5]))
Y_onehot       = prepare_output_onehot(Y[0:5])

print("lstm output shape:", lstm_output.shape)
print("Y onehot shape:", Y_onehot.shape)

lstm output shape: torch.Size([5, 32, 10])
Y onehot shape: torch.Size([5, 32, 10])


In [59]:
#Training LSTM w/ character embeddings. Feel free to change number of epochs, optimizer, learning rate and batch size.

nEpochs = 20

def train_char_lstm(sentences, tags, lstm):

    optimizer = optim.Adam(lstm.parameters(), lr=0.001)
    
    loss_fn = nn.NLLLoss()

    batchSize = 50

    for epoch in range(nEpochs):
        totalLoss = 0.0

        (sentences_shuffled, tags_shuffled) = shuffle_sentences(sentences, tags)
        for batch in tqdm(range(0, len(sentences), batchSize), leave=False):
            lstm.zero_grad()

            batch_sentences = sentences_shuffled[batch:batch+batchSize]
            batch_tags = tags_shuffled[batch:batch+batchSize]

            (X, X_mask, X_char) = lstm.sentences2input_tensors(batch_sentences)

            Y_gold = prepare_input(sentences2indices(batch_tags, tag2i))[0]

            X = X.cuda()
            X_char = X_char.cuda()
            Y_gold = Y_gold.cuda()


            log_probabilities = lstm.forward(X, X_char)

            log_probabilities = log_probabilities.view(-1, log_probabilities.shape[2])
            Y_gold = Y_gold.view(-1)

            loss = loss_fn(log_probabilities, Y_gold)

            loss.backward()
            optimizer.step()

            totalLoss += loss.item()

        print(f"loss on epoch {epoch} = {totalLoss}")
        lstm.write_predictions(sentences_dev, 'dev_pred')   #Performance on dev set
        print('conlleval:')
        print(subprocess.Popen('paste dev dev_pred | perl conlleval.pl -d "\t"', shell=True, stdout=subprocess.PIPE,stderr=subprocess.STDOUT).communicate()[0].decode('UTF-8'))

        if epoch % 10 == 0:
            s = sample(range(len(sentences_dev)), 5)
            lstm.print_predictions([sentences_dev[i] for i in s], [tags_dev[i] for i in s])

char_lstm = CharLSTMtagger(DIM_HID=500, DIM_EMB=300).cuda()
char_lstm.embedding.weight.data.copy_(lstm.embedding.weight.data.clone())
train_char_lstm(sentences_train, tags_train, char_lstm)

  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 0 = 23.254379062913358
conlleval:
processed 51578 tokens with 5942 phrases; found: 6254 phrases; correct: 5195.
accuracy:  97.70%; precision:  83.07%; recall:  87.43%; FB1:  85.19
              LOC: precision:  90.64%; recall:  91.24%; FB1:  90.94  1849
             MISC: precision:  70.65%; recall:  78.85%; FB1:  74.53  1029
              ORG: precision:  73.76%; recall:  84.27%; FB1:  78.66  1532
              PER: precision:  90.13%; recall:  90.23%; FB1:  90.18  1844

----------------------------
-START-/START/START Jakarta/I-LOC/I-LOC served/O/O as/O/O the/O/O host/O/O to/O/O the/O/O series/O/O of/O/O negotiations/O/O which/O/O culminated/O/O in/O/O the/O/O initialling/O/O of/O/O the/O/O agreement/O/O last/O/O Friday/O/O ./O/O -END-/END/END
Predicted:	 ['START', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'END']
Gold:		 ['START', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 

  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 1 = 4.694129615090787
conlleval:
processed 51578 tokens with 5942 phrases; found: 6117 phrases; correct: 5345.
accuracy:  98.28%; precision:  87.38%; recall:  89.95%; FB1:  88.65
              LOC: precision:  90.17%; recall:  94.39%; FB1:  92.23  1923
             MISC: precision:  83.58%; recall:  80.04%; FB1:  81.77  883
              ORG: precision:  80.66%; recall:  87.40%; FB1:  83.89  1453
              PER: precision:  91.55%; recall:  92.35%; FB1:  91.95  1858



  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 2 = 2.2111313201021403
conlleval:
processed 51578 tokens with 5942 phrases; found: 6147 phrases; correct: 5386.
accuracy:  98.39%; precision:  87.62%; recall:  90.64%; FB1:  89.11
              LOC: precision:  92.98%; recall:  95.16%; FB1:  94.05  1880
             MISC: precision:  72.27%; recall:  86.77%; FB1:  78.86  1107
              ORG: precision:  85.82%; recall:  86.20%; FB1:  86.01  1347
              PER: precision:  92.77%; recall:  91.31%; FB1:  92.04  1813



... (30 outputs left out)

  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 18 = 0.016193039003155718
conlleval:
processed 51578 tokens with 5942 phrases; found: 6127 phrases; correct: 5342.
accuracy:  98.13%; precision:  87.19%; recall:  89.90%; FB1:  88.52
              LOC: precision:  94.34%; recall:  92.54%; FB1:  93.43  1802
             MISC: precision:  71.26%; recall:  85.79%; FB1:  77.85  1110
              ORG: precision:  85.15%; recall:  85.09%; FB1:  85.12  1340
              PER: precision:  91.20%; recall:  92.83%; FB1:  92.01  1875



  0%|          | 0/300 [00:00<?, ?it/s]

loss on epoch 19 = 0.011031885139459519
conlleval:
processed 51578 tokens with 5942 phrases; found: 6174 phrases; correct: 5379.
accuracy:  98.19%; precision:  87.12%; recall:  90.53%; FB1:  88.79
              LOC: precision:  93.11%; recall:  94.23%; FB1:  93.67  1859
             MISC: precision:  70.78%; recall:  86.44%; FB1:  77.83  1126
              ORG: precision:  85.38%; recall:  86.20%; FB1:  85.79  1354
              PER: precision:  92.37%; recall:  92.02%; FB1:  92.19  1835



In [60]:
#Evaluation on test set
char_lstm.write_predictions(sentences_test, 'test_pred_cnn_lstm.txt')
!wget https://raw.githubusercontent.com/aritter/twitter_nlp/master/data/annotated/wnut16/conlleval.pl
!paste test test_pred_cnn_lstm.txt | perl conlleval.pl -d "\t"

--2025-03-27 10:24:33--  https://raw.githubusercontent.com/aritter/twitter_nlp/master/data/annotated/wnut16/conlleval.pl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12754 (12K) [text/plain]
Saving to: ‘conlleval.pl.9’

conlleval.pl.9      100%[===================>]  12.46K  --.-KB/s    in 0s      

2025-03-27 10:24:33 (74.8 MB/s) - ‘conlleval.pl.9’ saved [12754/12754]

processed 46666 tokens with 5648 phrases; found: 6115 phrases; correct: 4736.
accuracy:  96.27%; precision:  77.45%; recall:  83.85%; FB1:  80.52
              LOC: precision:  85.80%; recall:  88.37%; FB1:  87.06  1718
             MISC: precision:  47.50%; recall:  78.35%; FB1:  59.14  1158
              ORG: precision:  78.77%; recall:  80.19%; FB1:  79.47  1691
              PER: preci